**Repositories (Silver)**

In [0]:
repos_raw = spark.read.json(
    "/Volumes/workspace/default/github_data/raw/repos/*/"
)

repos_raw.printSchema()

root
 |-- allow_forking: boolean (nullable = true)
 |-- archive_url: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- assignees_url: string (nullable = true)
 |-- blobs_url: string (nullable = true)
 |-- branches_url: string (nullable = true)
 |-- clone_url: string (nullable = true)
 |-- collaborators_url: string (nullable = true)
 |-- comments_url: string (nullable = true)
 |-- commits_url: string (nullable = true)
 |-- compare_url: string (nullable = true)
 |-- contents_url: string (nullable = true)
 |-- contributors_url: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- custom_properties: struct (nullable = true)
 |    |-- allow_coworker_prs: string (nullable = true)
 |    |-- is_vendor_fork: string (nullable = true)
 |    |-- repo_protection_L2: string (nullable = true)
 |    |-- repo_protection_L3: string (nullable = true)
 |    |-- repo_protection_L3_KIKD: string (nullable = true)
 |    |-- repo_protection_L3_no_signing: string (nullabl

In [0]:
from pyspark.sql.functions import col

repos_silver = repos_raw.select(
    col("id").alias("repo_id"),
    col("name").alias("repo_name"),
    col("full_name"),
    col("owner.login").alias("owner"),
    col("description"),
    col("created_at"),
    col("updated_at"),
    col("pushed_at"),
    col("language"),
    col("stargazers_count").alias("stars"),
    col("forks_count").alias("forks"),
    col("open_issues_count").alias("open_issues"),
    col("default_branch"),
    col("ingestion_date")
)

In [0]:
repos_silver = repos_silver.dropDuplicates(["repo_id"])

repos_silver = repos_silver.fillna({
    "language": "Unknown",
    "description": "No description"
})

In [0]:
display(repos_silver)

repo_id,repo_name,full_name,owner,description,created_at,updated_at,pushed_at,language,stars,forks,open_issues,default_branch,ingestion_date
1100776768,claude-plugins-official,anthropics/claude-plugins-official,anthropics,"Official, Anthropic-managed directory of high quality Claude Code Plugins.",2025-11-20T18:36:20Z,2026-05-22T15:58:45Z,2026-05-22T15:48:25Z,Python,24181,2726,688,main,2026-05-22
1137078255,codegraph,colbymchenry/codegraph,colbymchenry,"Pre-indexed code knowledge graph for Claude Code, Codex, Cursor, and OpenCode — fewer tokens, fewer tool calls, 100% local",2026-01-18T21:45:37Z,2026-05-22T15:58:27Z,2026-05-22T10:58:36Z,TypeScript,15946,878,128,main,2026-05-22
1161163182,openhuman,tinyhumansai/openhuman,tinyhumansai,"Your Personal AI super intelligence. Private, Simple and extremely powerful.",2026-02-18T20:01:27Z,2026-05-22T15:58:48Z,2026-05-22T13:47:44Z,Rust,25536,2330,177,main,2026-05-22


In [0]:
repos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("repositories")

**Contributors (Silver)**

In [0]:
contributors_raw = spark.read.json(
    "/Volumes/workspace/default/github_data/raw/contributors/*/"
)

contributors_raw.printSchema()

root
 |-- avatar_url: string (nullable = true)
 |-- contributions: long (nullable = true)
 |-- events_url: string (nullable = true)
 |-- followers_url: string (nullable = true)
 |-- following_url: string (nullable = true)
 |-- gists_url: string (nullable = true)
 |-- gravatar_id: string (nullable = true)
 |-- html_url: string (nullable = true)
 |-- id: long (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- login: string (nullable = true)
 |-- node_id: string (nullable = true)
 |-- organizations_url: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- received_events_url: string (nullable = true)
 |-- repo: string (nullable = true)
 |-- repos_url: string (nullable = true)
 |-- site_admin: boolean (nullable = true)
 |-- starred_url: string (nullable = true)
 |-- subscriptions_url: string (nullable = true)
 |-- type: string (nullable = true)
 |-- url: string (nullable = true)
 |-- user_view_type: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

developers_silver = contributors_raw.select(
    col("id").alias("dev_id"),
    col("login"),
    col("contributions")
)

In [0]:
developers_silver = developers_silver.dropDuplicates(["dev_id"])

developers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/github_data/silver/developers/")

developers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("developers")

**REPO CONTRIBUTORS**

In [0]:
repo_contributors_silver = contributors_raw.select(
    col("id").alias("dev_id"),
    col("login").alias("dev_name"),
    col("repo"),
    col("owner"),
    col("contributions"),
    col("ingestion_date")
)

In [0]:
repo_contributors_silver = repo_contributors_silver.dropDuplicates(
    ["dev_id", "repo"]
)

repo_contributors_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("repo_contributors")

**Commits (Silver)**

In [0]:
commits_raw = spark.read.json(
    "/Volumes/workspace/default/github_data/raw/commits/*/"
)

commits_raw.printSchema()

root
 |-- author: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- events_url: string (nullable = true)
 |    |-- followers_url: string (nullable = true)
 |    |-- following_url: string (nullable = true)
 |    |-- gists_url: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- html_url: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- node_id: string (nullable = true)
 |    |-- organizations_url: string (nullable = true)
 |    |-- received_events_url: string (nullable = true)
 |    |-- repos_url: string (nullable = true)
 |    |-- site_admin: boolean (nullable = true)
 |    |-- starred_url: string (nullable = true)
 |    |-- subscriptions_url: string (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- user_view_type: string (nullable = true)
 |-- comments_url: string (nullable = true)
 |-- commit: struct

In [0]:
commits_silver = commits_raw.select(
    col("sha").alias("commit_id"),
    col("author.id").alias("dev_id"),
    col("author.login").alias("developer"),
    col("commit.author.date").alias("commit_timestamp"),
    col("commit.message").alias("message"),
    col("repo"),
    col("owner"),
    col("ingestion_date")
)

In [0]:
commits_silver = commits_silver.filter(col("dev_id").isNotNull())
commits_silver = commits_silver.dropDuplicates(["commit_id"])

In [0]:
# Convert Timestamp
from pyspark.sql.functions import to_timestamp

commits_silver = commits_silver.withColumn(
    "commit_timestamp",
    to_timestamp("commit_timestamp")
)

In [0]:
display(commits_silver)

commit_id dev_id developer commit_timestamp message repo owner ingestion_date 002dc8d76d45e96b8cbd0e1508b168f0036c5cd8 250988387 sanil-23 2026-05-22T09:59:03.000Z fix(subagent): dedup tool specs before sending to provider (#2485)

Co-authored-by: sanil-23 
Co-authored-by: Claude Opus 4.7 (1M context) openhuman tinyhumansai 2026-05-22 004775534b22bfb019f5490d8d4f899615f0dd5b 90404176 PranavAgarkar07 2026-05-17T03:21:12.000Z fix(security): truncate command in policy error message (closes #1941) (#1950) openhuman tinyhumansai 2026-05-22 00679aef889efe36bb0389f81d70b6229a2013ee 238056179 bryan-anthropic 2026-05-09T20:40:06.000Z Add sap-fiori-mcp-server plugin (#1777)

MCP server for SAP Fiori development tools — build and modify SAP Fiori
applications with AI assistance. Pinned at d9d4ab7e (latest main of
SAP/open-ux-tools). claude-plugins-official anthropics 2026-05-22 00b298966fe337b33a1f777354776c02816afe54 18431132 colbymchenry 2026-05-08T03:06:40.000Z docs(readme): add Initialize Projects snippet to Get Started (#145)

The top-level Get Started section showed the install command but not
the per-project init step. Adding the same `cd your-project /
codegraph init -i` block that lives in Quick Start so users see the
full happy path before scrolling.

Co-authored-by: Claude Opus 4.7 (1M context) codegraph colbymchenry 2026-05-22 00f13a5f46419a78b5cc1a344e890cb404843881 100382 dhollman 2026-03-10T20:21:32.000Z Merge pull request #106 from obahareth/main

Add Ruby LSP plugin with inline lspServers configuration claude-plugins-official anthropics 2026-05-22 013381e88094bd6d69ec9a2be5756662e78651d3 66018853 JAYcodr 2026-05-21T17:32:54.000Z fix(i18n): complete zh-CN translations for workspace, mascot, MCP Ser… (#2440)

Co-authored-by: agent:skill-master openhuman tinyhumansai 2026-05-22 01a8227827f131ad10f93bb2a38135774620c14a 46887634 aqilaziz 2026-05-20T23:45:54.000Z Quiet core-state and rewards timeout diagnostics

## Summary

- Limits core-state bootstrap console warnings to the first failure plus the existing backoff suppression notice.
- Moves rewards page diagnostics from unconditional `console.debug` calls to the namespaced `debug` logger.
- Normalizes `/rewards/me` timeout/abort errors into a stable recoverable UI message.
- Adds focused coverage for rewards timeout handling and the quieter core-state warning contract.

## Problem

- #1235 reports noisy DevTools output during startup and rewards loading.
- Core-state polling already had a retry budget/backoff, but still emitted repeated bootstrap warnings.
- Rewards timeout failures surfaced raw transport messages and unconditional debug logs.

## Solution

- Keep detailed per-attempt core-state diagnostics behind the existing `debug('core-state')` logger while reducing default console warnings.
- Wrap rewards API failures with `normalizeRewardsApiError`, preserving backend errors while mapping timeout/abort failures to a retryable message.
- Keep rewards snapshot retries manual-only through the existing Try again button.

## Submission Checklist

- [x] Tests added or updated (happy path + at least one failure / edge case) per [Testing Strategy](../gitbooks/developing/testing-strategy.md#failure-path-requirement)
- [x] **Diff coverage ≥ 80%** — changed lines (Vitest + cargo-llvm-cov merged via `diff-cover`) meet the gate enforced by [`.github/workflows/coverage.yml`](../.github/workflows/coverage.yml). Run `pnpm test:coverage` and `pnpm test:rust` locally; PRs below 80% on changed lines will not merge.
- [x] Coverage matrix updated — N/A: behavior-only frontend diagnostics change, no feature row added/removed/renamed.
- [x] All affected feature IDs from the matrix are listed in the PR description under `## Related`
- [x] No new external network dependencies introduced (mock backend used per [Testing Strategy](../gitbooks/developing/testing-strategy.md#mock-policy))
- [x] Manual smoke checklist updated if this touches release-cut surfaces — N/A: no release-cut surface touched

In [0]:
# Store Silver Layer
commits_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/github_data/silver/commits/")

# Register Table
commits_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("commits")

**Pull Requests (Silver)**

In [0]:
prs_raw = spark.read.json(
    "/Volumes/workspace/default/github_data/raw/pull_requests/*/"
)

prs_raw.printSchema()

root
 |-- _links: struct (nullable = true)
 |    |-- comments: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- commits: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- html: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- issue: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- review_comment: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- review_comments: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- self: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- statuses: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |-- active_lock_reason: string (nullable = true)
 |-- assignee: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- events_url: string (nullable = true)
 |    |-- followers_url: string (nullable = 

In [0]:
from pyspark.sql.functions import col

prs_silver = prs_raw.select(
    col("id").alias("pr_id"),
    col("number").alias("pr_number"),
    col("user.id").alias("dev_id"),
    col("user.login").alias("developer"),
    col("state"),
    col("title"),
    col("created_at"),
    col("updated_at"),
    col("closed_at"),
    col("merged_at"),
    col("merge_commit_sha"),
    col("repo"),
    col("owner"),
    col("ingestion_date")
)

In [0]:
prs_silver = prs_silver.filter(col("dev_id").isNotNull())
prs_silver = prs_silver.dropDuplicates(["pr_id"])

In [0]:
#Convert to timestamp
from pyspark.sql.functions import to_timestamp

prs_silver = prs_silver \
    .withColumn("created_at", to_timestamp("created_at")) \
    .withColumn("updated_at", to_timestamp("updated_at")) \
    .withColumn("closed_at", to_timestamp("closed_at")) \
    .withColumn("merged_at", to_timestamp("merged_at"))

In [0]:
display(prs_silver)

pr_id,pr_number,dev_id,developer,state,title,created_at,updated_at,closed_at,merged_at,merge_commit_sha,repo,owner,ingestion_date
3361444733,52,47449406,matthewhembree,open,Add Cursor IDE support to interactive installer,2026-03-06T00:12:40.000Z,2026-05-20T05:25:42.000Z,null,null,null,codegraph,colbymchenry,2026-05-22
3390335126,57,6419077,cfournel,open,add mql5 support,2026-03-12T15:33:24.000Z,2026-05-20T05:25:42.000Z,null,null,null,codegraph,colbymchenry,2026-05-22
3391492908,58,53652,malo,open,feat: add ReScript language support,2026-03-12T19:43:09.000Z,2026-05-20T05:25:42.000Z,null,null,null,codegraph,colbymchenry,2026-05-22
3426989416,831,100382,dhollman,open,"feat(telegram,discord): migrate bot tokens to plugin userConfig secrets",2026-03-20T21:56:13.000Z,2026-03-24T23:44:20.000Z,null,null,null,claude-plugins-official,anthropics,2026-05-22
3446404284,1005,12701358,noahzweben,open,Add Apache 2.0 LICENSE to math-olympiad plugin,2026-03-25T17:05:26.000Z,2026-03-30T17:17:37.000Z,null,null,null,claude-plugins-official,anthropics,2026-05-22
3455131775,1050,3757768,dicksontsai,open,Add version field to 27 local-sourced plugins,2026-03-27T05:02:53.000Z,2026-05-14T16:04:09.000Z,null,null,null,claude-plugins-official,anthropics,2026-05-22
3500053501,1287,12701358,noahzweben,open,rename(plugin-json): figma → figma-plugin,2026-04-07T20:53:06.000Z,2026-04-07T20:53:10.000Z,null,null,ff84188d18bcccc3d6d803688534ad2512d198df,claude-plugins-official,anthropics,2026-05-22
3535600562,1424,12701358,noahzweben,open,"fix(telegram): v0.0.7 reliability rollup — state-dir, PID guard, ppid watchdog, install stdout",2026-04-15T18:41:17.000Z,2026-05-17T13:17:48.000Z,null,null,b7d4aeea9cdd94480b2ac7768c1b8f552445d805,claude-plugins-official,anthropics,2026-05-22
3575334574,1561,269241295,markn-ant,open,Add managed-agents plugin,2026-04-23T20:00:25.000Z,2026-04-23T20:00:25.000Z,null,null,null,claude-plugins-official,anthropics,2026-05-22
3584953971,92,261684394,andreinknv,open,feat: add HCL / Terraform language support,2026-04-26T04:06:22.000Z,2026-05-20T05:25:42.000Z,null,null,null,codegraph,colbymchenry,2026-05-22


In [0]:
# Store Silver Layer
prs_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/github_data/silver/pull_requests/")

# Store Delta Table
prs_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pull_requests")